<a href="https://colab.research.google.com/github/victoria-banks/CCIS-301/blob/main/data_preprocessing_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing Assignment — Tasks 1–5

**Google Colab version**

### Upload these files when prompted
- `song_views.csv`
- `Pokemon.csv`
- `item_popularity.csv`
- `fcc_2016_coder_survey_subset.csv`
- `const.txt`

No dataset paths are hard-coded. The files are uploaded directly into the Colab runtime.

### Tasks
1. Binarization
2. Interaction, ordinal transformation, and one-hot encoding
3. Standard and Min-Max scaling
4. Fixed binning, adaptive binning, and log transform
5. Bag of Words and 2-gram model


In [ ]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_extraction.text import CountVectorizer

print("Libraries loaded successfully.")


## Upload all datasets

Run the cell below and select all five assignment files.


In [ ]:
# Upload all files
uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded.keys():
    print("-", filename)


In [ ]:
# Load uploaded datasets

required_files = [
    "song_views.csv",
    "Pokemon.csv",
    "item_popularity.csv",
    "fcc_2016_coder_survey_subset.csv",
    "const.txt"
]

missing_files = [f for f in required_files if f not in uploaded]

if missing_files:
    print("Missing file(s):")
    for f in missing_files:
        print("-", f)
    print("\nUpload the missing file(s) and rerun this cell.")
else:
    song = pd.read_csv("song_views.csv")
    pokemon = pd.read_csv("Pokemon.csv")
    item_popularity = pd.read_csv("item_popularity.csv")
    fcc = pd.read_csv("fcc_2016_coder_survey_subset.csv")

    with open("const.txt", "r", encoding="utf-8") as f:
        const_text = f.read()

    print("All datasets loaded successfully.")
    print("song_views:", song.shape)
    print("Pokemon:", pokemon.shape)
    print("item_popularity:", item_popularity.shape)
    print("FCC:", fcc.shape)


# Task 1 — Binarization

Convert `listen_count` into a binary variable:

- 0 → 0
- greater than 0 → 1


In [ ]:
# Task 1: Binarization

song_binary = song.copy()

song_binary["listen_binary"] = np.where(
    song_binary["listen_count"] > 0,
    1,
    0
)

display(song_binary.head(20))


In [ ]:
# Check Task 1 result

print("Binary value counts:")
print(song_binary["listen_binary"].value_counts())


**Screenshot:** Take a screenshot of the Task 1 output table above for submission.


# Task 2 — Pokémon Dataset

## 2A. Interaction


In [ ]:
# Interaction example: Attack × Defense

pokemon_interaction = pokemon.copy()

pokemon_interaction["Attack_Defense_Interaction"] = (
    pokemon_interaction["Attack"] *
    pokemon_interaction["Defense"]
)

display(
    pokemon_interaction[
        ["Name", "Attack", "Defense", "Attack_Defense_Interaction"]
    ].head(10)
)


## 2B. Transforming Ordinal Features

Generation has a natural order, so it is transformed as:

Gen 1 → 1, Gen 2 → 2, ..., Gen 6 → 6.


In [ ]:
# Ordinal transformation of Generation

pokemon_ordinal = pokemon.copy()

pokemon_ordinal["Generation_Ordinal"] = (
    pokemon_ordinal["Generation"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
    .astype("Int64")
)

display(
    pokemon_ordinal[
        ["Name", "Generation", "Generation_Ordinal"]
    ].head(10)
)


## 2C. One-Hot Encoding

One-hot encode the `Type 1` categorical feature.


In [ ]:
# One-hot encoding example

pokemon_onehot = pd.get_dummies(
    pokemon["Type 1"],
    prefix="Type1",
    dtype=int
)

pokemon_onehot_example = pd.concat(
    [
        pokemon[["Name", "Type 1"]],
        pokemon_onehot
    ],
    axis=1
)

display(pokemon_onehot_example.head(10))


# Task 3 — Standard and Min-Max Scaling

## 3A. song_views.csv


In [ ]:
# Identify numerical columns

song_numeric_columns = song.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical columns:")
print(song_numeric_columns)


In [ ]:
# Standard scaling and Min-Max scaling

song_scaled = song.copy()

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

song_scaled[
    [f"{col}_standard" for col in song_numeric_columns]
] = standard_scaler.fit_transform(
    song[song_numeric_columns]
)

song_scaled[
    [f"{col}_minmax" for col in song_numeric_columns]
] = minmax_scaler.fit_transform(
    song[song_numeric_columns]
)

display(song_scaled.head(10))


## 3B. item_popularity.csv


In [ ]:
# Identify numerical columns

item_numeric_columns = item_popularity.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical columns:")
print(item_numeric_columns)


In [ ]:
# Standard scaling and Min-Max scaling

item_scaled = item_popularity.copy()

item_standard_scaler = StandardScaler()
item_minmax_scaler = MinMaxScaler()

item_scaled[
    [f"{col}_standard" for col in item_numeric_columns]
] = item_standard_scaler.fit_transform(
    item_popularity[item_numeric_columns]
)

item_scaled[
    [f"{col}_minmax" for col in item_numeric_columns]
] = item_minmax_scaler.fit_transform(
    item_popularity[item_numeric_columns]
)

display(item_scaled.head(10))


### Scaling formulas

**Standard scaling**

`z = (x - mean) / standard deviation`

**Min-Max scaling**

`x_scaled = (x - minimum) / (maximum - minimum)`


# Task 4 — Fixed Binning, Adaptive Binning, and Log Transform


In [ ]:
# Show numerical columns in the FCC dataset

fcc_numeric_columns = fcc.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical columns:")
print(fcc_numeric_columns)


In [ ]:
# Use Income for binning and log transformation

income = pd.to_numeric(
    fcc["Income"],
    errors="coerce"
)

fcc_transformed = fcc[["Income"]].copy()

display(fcc_transformed["Income"].describe())


## 4A. Fixed Binning

Five equal-width bins are created.


In [ ]:
# Fixed-width binning

fixed_edges = np.linspace(
    income.min(),
    income.max(),
    6
)

fcc_transformed["Fixed_Bin"] = pd.cut(
    income,
    bins=fixed_edges,
    labels=[
        "Bin 1",
        "Bin 2",
        "Bin 3",
        "Bin 4",
        "Bin 5"
    ],
    include_lowest=True,
    duplicates="drop"
)

display(fcc_transformed.head(20))


## 4B. Adaptive Binning

Equal-frequency/quantile binning is used so that each bin contains approximately the same number of observations.


In [ ]:
# Adaptive/equal-frequency binning

fcc_transformed["Adaptive_Bin"] = pd.qcut(
    income,
    q=5,
    labels=[
        "Q1",
        "Q2",
        "Q3",
        "Q4",
        "Q5"
    ],
    duplicates="drop"
)

display(fcc_transformed.head(20))


## 4C. Log Transform


In [ ]:
# Log transformation

fcc_transformed["Income_Log"] = np.log1p(income)

display(fcc_transformed.head(20))


# Task 5 — Bag of Words and 2-Gram Model


## 5A. Bag of Words


In [ ]:
# Create Bag of Words

bow_vectorizer = CountVectorizer(
    lowercase=True
)

bow_matrix = bow_vectorizer.fit_transform(
    [const_text]
)

bow = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)

display(bow)


In [ ]:
# Show the 25 most frequent words

bow_frequencies = (
    bow.T
    .rename(columns={0: "Count"})
    .sort_values("Count", ascending=False)
)

display(bow_frequencies.head(25))


## 5B. 2-Gram Model


In [ ]:
# Create a 2-gram (bigram) model

bigram_vectorizer = CountVectorizer(
    lowercase=True,
    ngram_range=(2, 2)
)

bigram_matrix = bigram_vectorizer.fit_transform(
    [const_text]
)

bigrams = pd.DataFrame(
    bigram_matrix.toarray(),
    columns=bigram_vectorizer.get_feature_names_out()
)

display(bigrams)


In [ ]:
# Show the 25 most frequent 2-grams

bigram_frequencies = (
    bigrams.T
    .rename(columns={0: "Count"})
    .sort_values("Count", ascending=False)
)

display(bigram_frequencies.head(25))


# Export All Results

The following cell creates CSV files from the completed transformations.


In [ ]:
# Export results

song_binary.to_csv("Task1_Binarization.csv", index=False)

pokemon_interaction.to_csv(
    "Task2_Interaction.csv",
    index=False
)

pokemon_ordinal.to_csv(
    "Task2_Ordinal.csv",
    index=False
)

pokemon_onehot_example.to_csv(
    "Task2_OneHot.csv",
    index=False
)

song_scaled.to_csv(
    "Task3_Song_Scaling.csv",
    index=False
)

item_scaled.to_csv(
    "Task3_Item_Scaling.csv",
    index=False
)

fcc_transformed.to_csv(
    "Task4_Binning_Log.csv",
    index=False
)

bow_frequencies.to_csv(
    "Task5_Bag_of_Words.csv"
)

bigram_frequencies.to_csv(
    "Task5_2Gram.csv"
)

print("All task results have been exported successfully.")


# Assignment Complete

Run the notebook from top to bottom in Google Colab.

Keep the output tables visible for your screenshots/submission.
